In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


In [0]:
# Utilities 

In [0]:
%run /Workspace/Users/ashishbudz@gmail.com/databricks_pipeline/1_setup/utilities

In [0]:
# S3 storage folder path
base_path = "s3://sportsbar-dp-child-company-prac/orders"

In [0]:
# Setting up Widgets
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","orders","Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

landing = f"{base_path}/landing/"
processed = f"{base_path}/processed/"



In [0]:
# Reading data from the S3 storage
df = spark.read.options(header=True, inferSchema=True).csv(f"{landing}*.csv").withColumn("read_timestamp", F.current_timestamp()).select("*","_metadata.file_name", "_metadata.file_size")

In [0]:
display(df.limit(10))

In [0]:
df.write.format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")


In [0]:
bronze_df = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source}')

In [0]:
display(bronze_df.withColumn(
    'order_qty',
    F.when(F.column('order_qty').isNull(), F.lit(1))
    .otherwise(F.column('order_qty'))
))

In [0]:
display(bronze_df.select('order_qty').summary('mean').select('order_qty').collect()[0][0])

In [0]:
avg = bronze_df.select('order_qty').summary('mean').select('order_qty').collect()[0][0]
display(bronze_df.withColumn(
    'order_qty',
    F.when(F.column('order_qty').isNull(), F.lit(avg))
    .otherwise(F.column('order_qty'))
))

In [0]:
bronze_df = bronze_df.dropna(subset = ['order_qty'])

In [0]:
display(bronze_df.select("*").where(F.column("order_qty").isNull()))

In [0]:
# Checking date formats:
display(bronze_df.select('order_placement_date').distinct())

In [0]:
# Replacing Day-of-week from date before parsing to Date object
bronze_df = bronze_df.withColumn(
    "order_placement_date",
    F.regexp_replace(F.column("order_placement_date"), r"^[A-Za-z]+,\s*", "")
).withColumn(
    "order_placement_date",
    F.coalesce(

        F.try_to_date(F.column("order_placement_date"), "dd/MM/yyyy"),
        F.try_to_date(F.column("order_placement_date"), "dd-MM-yyyy"),
        F.try_to_date(F.column("order_placement_date"), "yyyy/MM/dd"),
        F.try_to_date(F.column("order_placement_date"), "yyyy-MM-dd"),
        F.try_to_date(F.column("order_placement_date"), "MMMM dd, yyyy")
    )
)



In [0]:
import pyspark
pyspark.__version__

In [0]:
# Moving Files from 'Landing' to 'Processed' in S3 Bucket
files = dbutils.fs.ls(f'{landing}')
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f'{processed}{file_info.name}',
        True
    )

In [0]:
display(bronze_df)

In [0]:
# Test: Filtering out records where customer ID is not a valid numerical
display(bronze_df.where(F.column("customer_id").rlike("^[0-9]+$")))

In [0]:
bronze_df = bronze_df.withColumn(
    "customer_id",
    F.when(F.column("customer_id").rlike("^[0-9]+$"), F.column("customer_id"))
    .otherwise(F.lit("999999"))
)

In [0]:
# Count before dropping duplicates 
display(bronze_df.count())

In [0]:
# Dropping Duplicates
bronze_df = bronze_df.dropDuplicates(['order_id','order_placement_date','customer_id','product_id','order_qty'])

In [0]:
# Count after dropping duplicates
bronze_df.count()

In [0]:
# Converting product id to String
bronze_df = bronze_df.withColumn(
    "product_id",
    F.column("product_id").cast("string")
)

In [0]:
bronze_df.printSchema()

In [0]:
# Minimum and maximum dates
display(bronze_df.agg(
    F.min(F.column("order_placement_date")).alias("min_date"),
    F.max(F.column("order_placement_date")).alias("max_date")
))

In [0]:
product_table = spark.table("fmcg.silver.products")

In [0]:
display(product_table)

In [0]:
df_joined = product_table.join(bronze_df, on="product_id", how="inner").select(bronze_df["*"], product_table["product_code"])

In [0]:
display(df_joined)

In [0]:
# Testing conditional statement which checks for existing table 
print(spark.catalog.tableExists(f'{catalog}.{bronze_schema}.{data_source}'))
print(spark.catalog.tableExists(f'{catalog}.{silver_schema}.{data_source}'))

In [0]:
if not (spark.catalog.tableExists(f'{catalog}.{silver_schema}.{data_source}')):
    df_joined.write.format("delta").option("delta.enableChangeDataFeed", True).option("mergeSchema", True).mode("overwrite").saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
else:
    silver_data = DeltaTable.forName(spark,f'{catalog}.{silver_schema}.{data_source}')
    silver_data.alias("silver").merge(df_joined.alias("bronze"), "silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id AND silver.order_id = bronze.order_id AND silver.order_placement_date = bronze.order_placement_date").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
print(spark.catalog.tableExists(f'{catalog}.{bronze_schema}.{data_source}'))
print(spark.catalog.tableExists(f'{catalog}.{silver_schema}.{data_source}'))

In [0]:
# Now from silver to gold, we will first look at compatibility of the tables
silver_df = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source}')

In [0]:
display(silver_df)

In [0]:
display(silver_df.count())

In [0]:
display(silver_df.groupBy(["order_placement_date","customer_id","product_id"]).agg({"order_qty":"sum"}).sort(["order_placement_date","customer_id"], ascending = [True,True]))

In [0]:
silver_df = silver_df.withColumnsRenamed({"order_placement_date":"date","customer_id":"customer_code","order_qty":"sold_quantity"})

In [0]:
silver_df = silver_df.select(["order_id","date","customer_code","product_code","sold_quantity"])

In [0]:
display(silver_df)

In [0]:
if not (spark.catalog.tableExists(f'{catalog}.{gold_schema}.sb_fact_{data_source}')):
    silver_df.write.format("delta").option("delta.enableChangeDataFeed", True).option("mergeSchema", True).mode("overwrite").saveAsTable(f'{catalog}.{gold_schema}.sb_fact_{data_source}')
else:
    gold = DeltaTable.forName(spark,f'{catalog}.{gold_schema}.sb_fact_{data_source}')
    gold.alias("gold").merge(silver_df.alias("silver"), "silver.order_id = gold.order_id AND silver.date = gold.date AND silver.customer_code = gold.customer_code AND silver.product_code = gold.product_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


In [0]:
df_child = spark.sql(f'SELECT * FROM {catalog}.{gold_schema}.sb_fact_{data_source}')

In [0]:
display(df_child)

In [0]:
display(df_child.agg(
    F.min(F.column("date")).alias("min_date"),
    F.max(F.column("date")).alias("max_date")
))

In [0]:
# Converting all Date, truncating 'day' to ensure that all records in the same month have the same date value
df_child = df_child.withColumn(
    "date",
    F.trunc(F.column("date"), "MM")
)

In [0]:
display(df_child.select("date").distinct().sort("date"))

In [0]:
# Now Group By and Aggregation operation. Group by date, customer_code, and last will be product_code. Then aggregate on the sold_quantity

df_child = df_child.groupBy(["date","customer_code","product_code"]).agg(
    F.sum(F.column("sold_quantity")).alias("sold_quantity")
).sort(["date","customer_code"])

In [0]:
display(df_child)

In [0]:
# Merging with Parent table
gold_parent = DeltaTable.forName(spark,f'{catalog}.{gold_schema}.fact_{data_source}')
gold_parent.alias("gold_parent").merge(df_child.alias("gold_child"), "gold_parent.date = gold_child.date AND gold_parent.customer_code = gold_child.customer_code AND gold_parent.product_code = gold_child.product_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


In [0]:
print(gold_schema) 